In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
#splitting

df = pd.read_csv('/content/train.csv')

X, y = df.drop('SalePrice', axis=1), df['SalePrice']
y = np.log1p(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [3]:
#make a custom age transformer

from sklearn.base import BaseEstimator, TransformerMixin

class AgeFeatureCreator(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()

        if 'YearBuilt' in X_copy.columns and 'YrSold' in X_copy.columns:
            X_copy['HouseAge'] = X_copy['YrSold'] - X_copy['YearBuilt']
            X_copy.drop(['YearBuilt', 'YrSold'], axis=1, inplace=True)

        if 'GarageYrBlt' in X_copy.columns:
            X_copy['GarageAge'] = X['YrSold'] - X_copy['GarageYrBlt'] # Use the original YrSold before dropping
            X_copy.drop('GarageYrBlt', axis=1, inplace=True)

        return X_copy

In [4]:
# Define the quality mapping
quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}

# List of columns where the quality is Ordinal
ordinal_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
                'HeatingQC', 'KitchenQual', 'FireplaceQu',
                'GarageQual', 'GarageCond', 'PoolQC']

# Apply the mapping to the dataframes
for col in ordinal_cols:
    X_train[col] = X_train[col].map(quality_map).fillna(0)
    X_test[col] = X_test[col].map(quality_map).fillna(0)

In [5]:
#Pipelining

numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include=['object']).columns

numerical_transformer = Pipeline(
    steps=[
        ('age_creator', AgeFeatureCreator()),
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

In [6]:
#fit once, transform everywhere

preprocessor.fit(X_train)

X_train_preprocessed = preprocessor.transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

In [7]:
# final verdict

from sklearn.metrics import mean_squared_error, r2_score

pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('regressor', LinearRegression())
    ]
)

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE:", rmse)

r2 = r2_score(y_test, y_pred)
print("R^2 Score:", r2)

RMSE: 0.12569963057417055
R^2 Score: 0.9153297603428929


In [8]:
import joblib

filename = 'house_price_pipeline_v1.joblib'
joblib.dump(pipeline, filename)

print(f"Pipeline successfully saved to {filename}")

Pipeline successfully saved to house_price_pipeline_v1.joblib
